# Seminar 9: Autoregression, VAE, and GAN

**Deep Learning Course 2026**

**Author:** Nikita Kiselev

In [ ]:
import math
import torch
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["font.size"] = 16

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

## 1. Autoregression

Let's use binarized MNIST: every pixel is 0 or 1, so each conditional $p(x_i \mid \mathbf{x}_{<i})$ is a Bernoulli distribution.

We also load the test split to measure the likelihood.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    lambda x: (x > 0.5).float()
])
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=512)

**Building the model**

Image pixels are treated as input tokens. Their values can be either 0 or 1.

To leverage autoregressive modeling, we add `<START>` token into beginning, let it equals 2.

As we follow classification objective and use tokens from predefined vocabulary (0, 1, 2), we need embeddings for them.

Recurrent neural network is implemented with LSTM module.

The last layer projects hidden states into probabilities of ones per each pixel.

In [ ]:
class PixelRNN(nn.Module):
    def __init__(self, hidden_dim=512):
        super().__init__()
        self.embed = nn.Embedding(3, hidden_dim)
        self.rnn = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        start_token = torch.full((x.shape[0], 1), 2, dtype=torch.long, device=x.device)
        x = torch.cat([start_token, x[:, :-1]], dim=1)  # [b, 784], shifted right
        x = self.embed(x)  # [b, 784, d]
        x, _ = self.rnn(x)  # [b, 784, d], batch_first=True
        x = self.fc(x)  # [b, 784, 1], logits
        return x

**Training and evaluating the model**

We've already built the model class, so let's train it!

Firstly, we define a loss function – Binary Cross Entropy.

It is exactly the negative log-likelihood of a Bernoulli pixel, so we train by the exact maximum likelihood from the lecture.

In [ ]:
criterion = nn.BCEWithLogitsLoss()

Then, we make a training loop.

`criterion` averages over pixels, so dividing by $\ln 2$ reports the NLL in bits per dimension (pixel).

In [ ]:
epochs = 10
learning_rate = 3e-4

model = PixelRNN().to(device)
optimizer = optim.AdamW(model.parameters(), learning_rate)

model.train()

for epoch in range(epochs):
    total = 0
    for x, _ in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
        x = x.view(-1, 784).to(device)
        logits = model(x.long())
        loss = criterion(logits.view(-1), x.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * x.shape[0]
    print(f"Epoch {epoch + 1}: train NLL {total / len(train_dataset) / math.log(2):.3f} bits/dim")

**Test likelihood**

Autoregressive models give the exact log-likelihood $\log p_{\boldsymbol{\theta}}(\mathbf{x}) = \sum_i \log p_{\boldsymbol{\theta}}(x_i \mid \mathbf{x}_{<i})$, so we can compute it on the test set.

In [ ]:
@torch.no_grad()
def nll_pixelrnn(model, loader):
    model.eval()
    total = 0
    for x, _ in loader:
        x = x.view(-1, 784).to(device)
        logits = model(x.long()).view(-1, 784)
        total += F.binary_cross_entropy_with_logits(logits, x, reduction="sum").item()
    return total / len(loader.dataset) / (784 * math.log(2))

nll_ar = nll_pixelrnn(model, test_loader)
print(f"PixelRNN test NLL: {nll_ar:.3f} bits/dim")

**Sampling pixel by pixel**

Generation is sequential: we start from the `<START>` token, predict the probability of the next pixel, sample it from a Bernoulli distribution and feed it back, carrying the LSTM state over all 784 steps.

In [ ]:
@torch.no_grad()
def sample(model, num_samples):
    model.eval()
    x = torch.full((num_samples, 1), 2, dtype=torch.long, device=device)
    h = c = None
    generated = []
    
    for _ in range(784):
        emb = model.embed(x)
        if h is None or c is None:
            out, (h, c) = model.rnn(emb)
        else:
            out, (h, c) = model.rnn(emb, (h, c))
        logits = model.fc(out)
        probs = torch.sigmoid(logits).squeeze(-1)
        x = torch.bernoulli(probs).long()
        generated.append(x)
        
    images = torch.stack(generated, dim=1)
    images = images.view(num_samples, 28, 28).float().cpu()
    return images

In [ ]:
num_samples = 16

images = sample(model, num_samples)

fig, axes = plt.subplots(2, num_samples // 2, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i], cmap="gray")
    ax.axis("off")
    
plt.suptitle("Generated Samples")
plt.show()

## 2. VAE

We use the same binarized MNIST as for PixelRNN, so we can compare the two models on the same test set.

**Building the encoder class**

The encoder takes an input image and compresses it into a compact latent representation.

But unlike a regular autoencoder, it doesn’t output a single point - instead, it outputs two vectors: the mean $\boldsymbol{\mu}$ and log-variance $\log \boldsymbol{\sigma}^2$.

These define a probability distribution from which we’ll later sample a latent vector.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.GELU()
        
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
    def forward(self, x):
        x = self.act(self.fc1(x))
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

**Implementing the decoder class**

The decoder does the reverse, it takes a sampled point from the latent space and tries to reconstruct the original image.

It outputs the parameters of $p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z})$: logits of the Bernoulli probabilities $p_j = \sigma(\text{logit}_j)$, one per pixel.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=20, hidden_dim=400, output_dim=784):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        return self.fc2(self.act(self.fc1(z)))  # logits

**Creating the main VAE class**

The VAE class combines the encoder and decoder and implements the reparameterization trick to keep training differentiable.

Since sampling directly from a distribution $\mathbf{z} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\sigma}^2)$ breaks backpropagation, we sample $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and compute $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \cdot  \boldsymbol{\epsilon}$ instead.

This trick allows gradients to flow through the sampling step during training.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(mu)
        z = mu + std * eps
        return z

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decoder(z)
        return logits, mu, logvar

**Loss function**

Training a VAE involves optimizing a composite loss function that balances two goals:

1. **Reconstruction Loss:** Ensures the output is close to the original input.

2. **Regularization Loss (KL divergence):** Encourages the learned latent distribution to be close to a standard normal distribution.

Together they form the negative ELBO (Evidence Lower Bound), which we minimize.

Pixels are binary, so $p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z}) = \prod_j \text{Bernoulli}(x_j \mid p_j)$, and the reconstruction loss is the binary cross-entropy:

$$ \mathcal{L}_{\text{rec}}(\boldsymbol{\theta}, \boldsymbol{\phi}) = - \sum_j \left[ x_j \log p_j + (1 - x_j) \log (1 - p_j) \right] $$

For grayscale images one would choose a Gaussian likelihood instead, which gives MSE.

Regularization loss is easily derived for gaussian latent distribution as follows:

$$ \mathcal{L}_{\text{KL}}(\boldsymbol{\phi}) = - \frac{1}{2} \sum_{j} \left( 1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2 \right) $$

Both terms are summed, not averaged: they are parts of one bound, so their relative weight is fixed by the math.

In [ ]:
def loss_function(x, logits, mu, logvar):
    recon_loss = F.binary_cross_entropy_with_logits(logits, x, reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss  # negative ELBO, summed over the batch

**Training and evaluating the VAE**

Now that we’ve defined the VAE model and its loss function, it’s time to train it.

We’ll run the training loop for several epochs, calculate the loss, and visualize how well the VAE learns to reconstruct and generate data.

In [ ]:
epochs = 30
learning_rate = 1e-3

model = VAE().to(device)
optimizer = optim.AdamW(model.parameters(), learning_rate)

model.train()

for epoch in range(epochs):
    total = 0
    for x, _ in train_loader:
        x = x.view(-1, 784).to(device)
        logits, mu, logvar = model(x)
        loss = loss_function(x, logits, mu, logvar)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f"Epoch {epoch + 1}: train -ELBO {total / len(train_dataset) / (784 * math.log(2)):.3f} bits/dim")

**Test bound**

For the VAE we cannot compute $\log p_{\boldsymbol{\theta}}(\mathbf{x})$ exactly, only the ELBO.

Since $\log p_{\boldsymbol{\theta}}(\mathbf{x}) \ge \mathcal{L}_{\boldsymbol{\theta}, \boldsymbol{\phi}}(\mathbf{x})$, the test $-\text{ELBO}$ is an **upper bound** on the VAE's NLL.

We estimate it with one sample of $\mathbf{z}$ per image.

In [ ]:
@torch.no_grad()
def negative_elbo(model, loader):
    model.eval()
    total = 0
    for x, _ in loader:
        x = x.view(-1, 784).to(device)
        logits, mu, logvar = model(x)
        total += loss_function(x, logits, mu, logvar).item()
    return total / len(loader.dataset) / (784 * math.log(2))

nelbo_vae = negative_elbo(model, test_loader)
print(f"PixelRNN test NLL: {nll_ar:.3f} bits/dim (exact)")
print(f"VAE test -ELBO:    {nelbo_vae:.3f} bits/dim (upper bound on NLL)")

What can we conclude?

The PixelRNN number is exact, the VAE number is only a bound: the true VAE NLL is lower by $\mathrm{KL}\big(q_{\boldsymbol{\phi}}(\mathbf{z} \mid \mathbf{x}) \,\|\, p_{\boldsymbol{\theta}}(\mathbf{z} \mid \mathbf{x})\big)$ on average.

So "PixelRNN < VAE" in this table does not strictly prove that PixelRNN has the better likelihood; tighter estimates (importance weighting) close part of the gap.

Papers use dynamically binarized MNIST, so these numbers are not directly comparable to published ones.

**Sampling from the latent space**

We’ll sample random points from a standard normal distribution and feed them through the decoder to generate synthetic images.

The decoder gives pixel probabilities $p_j$; we show them (the mean of $p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z})$) rather than a binary sample, which would look noisy.

In [ ]:
@torch.no_grad()
def sample(model, num_samples):
    model.eval()
    z = torch.randn(num_samples, 20).to(device)
    x = torch.sigmoid(model.decoder(z)).cpu()
    x = x.view(-1, 1, 28, 28)
    return x

In [ ]:
num_samples = 16

images = sample(model, num_samples)

fig, axes = plt.subplots(2, num_samples // 2, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i][0], cmap="gray")
    ax.axis("off")
    
plt.suptitle("Generated Samples")
plt.show()

**Latent space interpolation**

We can also interpolate between two points in the latent space to see smooth transitions in generated images, which is a hallmark of a well-trained VAE.

In [ ]:
@torch.no_grad()
def interpolate(model, z1, z2, steps=10):
    model.eval()
    z = torch.stack([
        z1 * (1 - t) + z2 * t for t in torch.linspace(0, 1, steps)
    ]).to(device)
    x = torch.sigmoid(model.decoder(z)).cpu()
    x = x.view(-1, 1, 28, 28)
    return x

In [ ]:
z1 = torch.randn(1, 20, generator=torch.Generator("cpu").manual_seed(0))
z2 = torch.randn(1, 20, generator=torch.Generator("cpu").manual_seed(1))

images = interpolate(model, z1, z2)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i][0], cmap="gray")
    ax.axis("off")
    
plt.suptitle("Latent Space Interpolation")
plt.show()

## 3. GAN

Unlike PixelRNN and the VAE, a GAN never evaluates a likelihood of pixels, so we can use grayscale (non-binarized) MNIST directly: the generator outputs intensities in $[0, 1]$.

In [ ]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)

**Building the Generator class**

Generator $G$ maps random noise $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ to images and tries to fool the discriminator.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=20, hidden_dim=400, output_dim=784):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.act1 = nn.GELU()
        
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.act2 = nn.Sigmoid()
        
    def forward(self, x):
        x = self.act1(self.fc1(x))
        x = self.act2(self.fc2(x))
        return x

**Implementing the Discriminator class**

Discriminator $D$ receives an image and outputs a logit $a$; the probability that the image is real (from data) rather than fake (from $G$) is $D(\mathbf{x}) = \sigma(a)$.

We keep the sigmoid inside the loss (`BCEWithLogitsLoss`), which computes $\log \sigma(a)$ and $\log (1 - \sigma(a))$ in a numerically stable way.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))  # logits

**Loss function**

We formalize GAN's training with a minimax objective:
$$
\min_{G} \max_{D} \left[ \mathbb{E}_{\pi(\mathbf{x})} \log D(\mathbf{x}) + \mathbb{E}_{p(\mathbf{z})} \log (1 - D(G(\mathbf{z}))) \right]
$$

Consider a binary cross entropy (BCE):
$$
\text{BCE}(\hat{y}, y) = - \left[ y \cdot \log \hat{y} + (1 - y) \cdot \log (1 - \hat{y}) \right]
$$

Using BCE, the discriminator loss (calculated on minibatch) becomes:

$$
\mathcal{L}_D = \text{BCE}(D(\mathbf{x}), 1) + \text{BCE}(D(G(\mathbf{z})), 0)
$$

If we followed the original minimax formula, the generator would minimize:

$$
\mathbb{E}_{p(\mathbf{z})} \log (1 - D(G(\mathbf{z})))
$$

But this saturates when $D$ is good at rejecting fakes, leading to weak gradients.

Instead, we use the **non-saturating trick**: maximize $\log D(G(\mathbf{z}))$, or in BCE minibatch notation:

$$
\mathcal{L}_G = \text{BCE}(D(G(\mathbf{z})), 1)
$$

**Training and evaluating the GAN**

Now that we’ve defined the GAN, it’s time to train it.

Two details matter:

1. Each network's gradients are zeroed right before its own backward pass.
2. The discriminator is frozen during the generator step, so the generator loss does not update $D$.

We also track $D(\mathbf{x})$, $D(G(\mathbf{z}))$ and $\mathcal{L}_D$: at the equilibrium from the lecture $D \equiv 0.5$ and $\mathcal{L}_D = 2 \log 2 \approx 1.386$.

In [ ]:
epochs = 50
learning_rate = 2e-4

generator = Generator().to(device)
discriminator = Discriminator().to(device)

optimizer_generator = optim.Adam(generator.parameters(), learning_rate, betas=(0.5, 0.999))
optimizer_discriminator = optim.Adam(discriminator.parameters(), learning_rate, betas=(0.5, 0.999))

criterion = nn.BCEWithLogitsLoss()
history = {"D(x)": [], "D(G(z))": [], "loss_D": []}

for epoch in tqdm(range(epochs), desc="Epoch"):
    stats = torch.zeros(3)
    for x_real, _ in train_loader:
        x_real = x_real.view(-1, 784).to(device)
        bs = x_real.shape[0]

        labels_real = torch.ones(bs, 1, device=device)
        labels_fake = torch.zeros(bs, 1, device=device)

        # Discriminator step
        z = torch.randn(bs, 20, device=device)
        x_fake = generator(z).detach()  # NOTE: stop gradient for generator here
        logits_real = discriminator(x_real)
        logits_fake = discriminator(x_fake)
        loss_discriminator = criterion(logits_real, labels_real) + criterion(logits_fake, labels_fake)
        optimizer_discriminator.zero_grad()
        loss_discriminator.backward()
        optimizer_discriminator.step()

        # Generator step, non-saturating loss
        discriminator.requires_grad_(False)  # NOTE: generator loss must not update D
        z = torch.randn(bs, 20, device=device)
        loss_generator = criterion(discriminator(generator(z)), labels_real)
        optimizer_generator.zero_grad()
        loss_generator.backward()
        optimizer_generator.step()
        discriminator.requires_grad_(True)

        stats += torch.stack([
            torch.sigmoid(logits_real).mean(),
            torch.sigmoid(logits_fake).mean(),
            loss_discriminator,
        ]).detach().cpu()

    for key, value in zip(history, stats / len(train_loader)):
        history[key].append(value.item())

**Does the game reach the equilibrium?**

Compare the curves with the theory: how far are they from $0.5$ and $2 \log 2$, and why?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history["D(x)"], label=r"$D(\mathbf{x})$")
axes[0].plot(history["D(G(z))"], label=r"$D(G(\mathbf{z}))$")
axes[0].axhline(0.5, color="gray", linestyle="--")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["loss_D"], label=r"$\mathcal{L}_D$")
axes[1].axhline(2 * math.log(2), color="gray", linestyle="--", label=r"$2 \log 2$")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.show()

**Sampling from the latent space**

We’ll sample random points from a standard normal distribution and feed them through the generator to generate synthetic images.

In [ ]:
@torch.no_grad()
def sample(generator, num_samples):
    generator.eval()
    z = torch.randn(num_samples, 20).to(device)
    x = generator(z).cpu()
    x = x.view(-1, 1, 28, 28)
    return x

In [ ]:
num_samples = 16

images = sample(generator, num_samples)

fig, axes = plt.subplots(2, num_samples // 2, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i][0], cmap="gray")
    ax.axis("off")
    
plt.suptitle("Generated Samples")
plt.show()

**Latent space interpolation**

We can also interpolate between two points in the latent space to see smooth transitions in generated images, since the generator is a smooth function of $\mathbf{z}$.

In [ ]:
@torch.no_grad()
def interpolate(generator, z1, z2, steps=10):
    generator.eval()
    z = torch.stack([
        z1 * (1 - t) + z2 * t for t in torch.linspace(0, 1, steps)
    ]).to(device)
    x = generator(z).cpu()
    x = x.view(-1, 1, 28, 28)
    return x

In [ ]:
z1 = torch.randn(1, 20, generator=torch.Generator("cpu").manual_seed(1))
z2 = torch.randn(1, 20, generator=torch.Generator("cpu").manual_seed(8))

images = interpolate(generator, z1, z2)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i][0], cmap="gray")
    ax.axis("off")
    
plt.suptitle("Latent Space Interpolation")
plt.show()